In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Load Model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
teacher_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
teacher_model.set_use_module_grafting(False)
teacher_model.set_use_scc_status(False)

In [27]:
outputs = teacher_model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.6582, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [28]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
teacher_model.to(device)

device

device(type='cuda')

### Load Trained Weights

In [29]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [30]:
from train_eval_func import eval_loop

In [31]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [32]:
teacher_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [33]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

# 5. Layer Score Importance And Sorting

In [34]:
from structure_prune_helper import compute_layer_importance_scores, register_importance_masks, remove_importance_masks

In [35]:
register_importance_masks(teacher_model, device, register_mha_layer=True, register_mlp_layer=True,
                         register_mha=True, register_mlp=True)

  Layer 0: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 1: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 2: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 3: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 4: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 5: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 6: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 7: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 8: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 9: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 10: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 11: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

# 6. Function for Grafting (Add-On)

In [36]:
from CKA import CKAEvaluator

cka_eval = CKAEvaluator(device)

In [43]:
for iter in range(6):
    print("Iteration:", iter)

    ### calculate layer similarity
    cls_reps_similarity = cka_eval.pairwise(
        model, 
        train_dataloader, 
        device, 
        only_cls_token=True, 
        max_iter=130
    )
    stats = cka_eval.similarity_stats(cls_reps_similarity)
    
    print("  Similarity Stats:")
    print(f"    Average: {stats['average']:.6f}")
    print(f"    Highest: {stats['max']:.6f}")
    print(f"    Lowest:  {stats['min']:.6f}")

    ## identify high similar pairs
    adj_similarities = []
    for i in range(1, len(cls_reps_similarity)):
        adj_similarities.append(cls_reps_similarity[i-1][i])

    high_pairs = []
    for idx, pair in enumerate(adj_similarities):
        if pair > 0.90:
            high_pairs.append(idx)
    
    print("Number of High Similar Pairs:", len(high_pairs))

    ## sensitivity score calculation
    mha_imp, ffn_imp, head_imp, neuron_imp, sublayer_imp = compute_layer_importance_scores(
        teacher_model, train_dataloader, device,
        compute_mha=True, compute_ffn=True,
        compute_heads=True, compute_neurons=True,
        display_separate=False, display_combined=False,
        display_pruning_order= False
    )

    print("")
    break

Iteration: 0


CKA Evaluation: 100%|██████████| 115/115 [00:25<00:00,  4.56it/s]


  Similarity Stats:
    Average: 0.899711
    Highest: 0.993545
    Lowest:  0.727337
Number of High Similar Pairs: 6


Computing importance (MHA layer, FFN layer, heads, neurons): 100%|██████████| 115/115 [00:25<00:00,  4.47it/s]

In [44]:
adj_similarities

[0.78567785,
 0.9596914,
 0.80230695,
 0.8586488,
 0.72733736,
 0.95349514,
 0.97812164,
 0.87347645,
 0.9745218,
 0.9935451,
 0.9899991]

In [39]:
mha_imp

array([0.90087897, 1.        , 0.5549724 , 0.45073274, 0.47345626,
       0.93563807, 0.24631655, 0.19929044, 0.23926191, 0.19579041,
       0.11616153, 0.06435577], dtype=float32)

In [45]:
mha_imp

array([0.73486745, 0.98318785, 0.61819506, 0.5084665 , 0.42395195,
       1.        , 0.22342065, 0.2024329 , 0.32459202, 0.2457308 ,
       0.14135739, 0.06339239], dtype=float32)

In [41]:
def reset_importance_masks(model,
                           reset_mha=True, reset_mlp=True,
                           reset_mha_layer=True, reset_mlp_layer=True):
    """
    Reset importance masks back to ones (identity) after computation.
    Masks remain registered as nn.Parameters but gradients are zeroed
    and values reset to 1.0 so they have no effect on forward pass.
    
    Call this after compute_layer_importance_scores() before training
    or before re-running importance computation.
    
    Args:
        model           : BERT model
        reset_mha       : reset per-head mask (head_mask_param)
        reset_mlp       : reset per-neuron mask (int_mask_param)
        reset_mha_layer : reset scalar MHA layer gate (head_layer_mask_param)
        reset_mlp_layer : reset scalar FFN layer gate (mlp_mask)
    """
    if not any([reset_mha, reset_mlp, reset_mha_layer, reset_mlp_layer]):
        raise ValueError("Must reset at least one mask type")

    for layer_idx, layer in enumerate(model.bert.encoder.layer):
        parts = []

        # --- reset per-head mask ---
        if reset_mha:
            if hasattr(layer, 'head_mask_param') and layer.head_mask_param is not None:
                with torch.no_grad():
                    layer.head_mask_param.fill_(1.0)
                layer.head_mask_param.grad = None
                parts.append(f"head_mask_param ({layer.head_mask_param.shape[0]} heads)")

        # --- reset per-neuron mask ---
        if reset_mlp:
            if hasattr(layer, 'int_mask_param') and layer.int_mask_param is not None:
                with torch.no_grad():
                    layer.int_mask_param.fill_(1.0)
                layer.int_mask_param.grad = None
                parts.append(f"int_mask_param ({layer.int_mask_param.shape[0]} neurons)")

        # --- reset scalar MHA gate ---
        if reset_mha_layer:
            if hasattr(layer, 'head_layer_mask_param') and layer.head_layer_mask_param is not None:
                with torch.no_grad():
                    layer.head_layer_mask_param.fill_(1.0)
                layer.head_layer_mask_param.grad = None
                parts.append("head_layer_mask_param")

        # --- reset scalar FFN gate ---
        if reset_mlp_layer:
            if hasattr(layer, 'mlp_mask') and layer.mlp_mask is not None:
                with torch.no_grad():
                    layer.mlp_mask.fill_(1.0)
                layer.mlp_mask.grad = None
                parts.append("mlp_mask")

        if parts:
            print(f"  Layer {layer_idx}: Reset ({', '.join(parts)})")
        else:
            print(f"  Layer {layer_idx}: Nothing to reset")

    return model

In [42]:
reset_importance_masks(model)

  Layer 0: Nothing to reset
  Layer 1: Nothing to reset
  Layer 2: Nothing to reset
  Layer 3: Nothing to reset
  Layer 4: Nothing to reset
  Layer 5: Nothing to reset
  Layer 6: Nothing to reset
  Layer 7: Nothing to reset
  Layer 8: Nothing to reset
  Layer 9: Nothing to reset
  Layer 10: Nothing to reset
  Layer 11: Nothing to reset


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,